## Initialization


In [68]:
from math import pi, sqrt, floor
from itertools import product

from qiskit import QuantumCircuit
from qiskit.circuit.library import WeightedAdder
from qiskit.quantum_info import Statevector

# Subset Sum with Grover Search — `WeightedAdder` Version

This notebook implements a **correct Grover oracle** for Subset Sum using Qiskit's `WeightedAdder`.



So this notebook answers:

> How do we solve a small Subset Sum instance with Grover search in Qiskit？

In this case, we search over bitstrings

$$x\in\{0,1\}^n.$$

For subset sum, define

$$f(x)=
\begin{cases}
1, & \sum_i x_i a_i = T,\\
0, & \text{otherwise}.
\end{cases}$$

The phase oracle is:

$$O_f|x\rangle = (-1)^{f(x)}|x\rangle$$

So:

$$O_f|x\rangle =
\begin{cases}
-|x\rangle, & x \text{ is a solution},\\
|x\rangle, & x \text{ is not a solution}.
\end{cases}
$$

Now we follow the usual Grover procedure: apply $G = D O_f$, where $D$ is the diffusion operator. The circuit has no classical bits and no measurements; after building the circuit, we use `Statevector` outside the circuit to inspect selector probabilities. 

However, the tricky thing here is the construction of the sum state $\ket{S(x)}$, where $S(x) = \sum_i x_i a_i$
In this project, we are going to construct a seperate register for the sum, which is represented via binary. The computation of the sum is done via weighted_adder in the qiskit library. If we have time, we will talk about a paper that detail the algorithm of summing up. But for now it is enough for work.  

## Demo example

We use

$$
S=\{1,2,3\}, \qquad T=3.
$$

The valid subsets are:
$$
\{3\}, \qquad \{1,2\}.
$$

So the marked selector bitstrings in internal order $(x_0,x_1,x_2)$ are:

$$
(0,0,1), \qquad (1,1,0).
$$

In [69]:
S = [1, 2, 3]
T = 3
n = len(S)

solutions = []
for bits in product([0, 1], repeat=n):
    if sum(bit * a for bit, a in zip(bits, S)) == T:
        solutions.append(bits)

print("S =", S)
print("T =", T)
print("Classical solutions in x[0],x[1],x[2] order:", solutions)

S = [1, 2, 3]
T = 3
Classical solutions in x[0],x[1],x[2] order: [(0, 0, 1), (1, 1, 0)]


## What `WeightedAdder` does

`WeightedAdder(weights=S)` constructs a reversible arithmetic circuit that computes:

$$ |x\rangle|0\rangle \mapsto |x\rangle\left|\sum_i x_i a_i\right\rangle $$

That is exactly the weighted-sum part needed for Subset Sum. We make use of it in the function subset_sum_oracle_weighted_adder.



Note this is a blockbox method and it comes with several helper qubits. So for the sake of optimization 

In [70]:
def little_endian_bits(value: int, width: int):
    '''
    It returns the little-endian bit representation of an integer value as a list of bits of given width.
    like 3 to [1, 1, 0] with width 3, or 5 to [1, 0, 1] with width 3.
    If the value is larger than 2**width - 1, it will return only the least significant bits.
    '''
    return [(value >> j) & 1 for j in range(width)]


def phase_flip_on_sum_equal_target(qc, sum_qubits, target):
    '''
    Apply phase -1 when the sum register equals target.

    This is a direct phase oracle:
        |sum> -> -|sum> if sum == target.
    '''
    target_bits = little_endian_bits(target, len(sum_qubits))

    # Convert target pattern to all-ones.
    for q, bit in zip(sum_qubits, target_bits):
        if bit == 0:
            qc.x(q)

    # Phase flip all-ones.
    if len(sum_qubits) == 1:
        qc.z(sum_qubits[0])
    else:
        qc.mcp(pi, sum_qubits[:-1], sum_qubits[-1])

    # Undo conversion.
    for q, bit in zip(sum_qubits, target_bits):
        if bit == 0:
            qc.x(q)


def subset_sum_oracle_weighted_adder(S, target):
    '''
    High-level subset-sum phase oracle using Qiskit's WeightedAdder.

    It computes sum_i x_i a_i, phase-flips if the sum equals target,
    then uncomputes the sum.
    '''
    n = len(S)
    adder = WeightedAdder(num_state_qubits=n, weights=S)

    qc = QuantumCircuit(adder.num_qubits, name="SubsetSumOracle")

    # Compute weighted sum into adder's sum register.
    qc.append(adder.to_gate(), range(adder.num_qubits))

    # In WeightedAdder, the first n qubits are state qubits.
    # The next adder.num_sum_qubits are sum qubits.
    sum_start = n
    sum_qubits = list(range(sum_start, sum_start + adder.num_sum_qubits))

    phase_flip_on_sum_equal_target(qc, sum_qubits, target)

    # Uncompute weighted sum.
    qc.append(adder.inverse().to_gate(), range(adder.num_qubits))

    return qc.to_gate()

**Note.**

`phase_flip_on_sum_equal_target` uses a multi-controlled phase gate, which naturally marks only the all-ones state of the sum register.

For example, with three sum qubits, the direct controlled phase marks:

$$
\ket{111}
$$

But the target sum is not always represented by the all-ones state. If the target is `T = 3`, then in little-endian qubit order its bits are:

$$
3 = [1, 1, 0]
$$

So before applying the controlled phase, we temporarily apply `X` gates to the qubits where the target bit is `0`. This maps the target pattern to all ones:

$$
[1, 1, 0] \longrightarrow [1, 1, 1]
$$

Then the multi-controlled phase flips exactly that state. After the phase flip, we apply the same `X` gates again to undo the temporary change.

This is a reversible basis change. It does not permanently modify the sum register; it only lets us reuse the simple “mark all-ones” controlled phase gate to mark an arbitrary target value.

## Diffusion operator


In [71]:
def diffusion_on_first_n_qubits(n, total_qubits):
    qc = QuantumCircuit(total_qubits, name="Diffusion")

    xs = list(range(n))

    qc.h(xs)
    qc.x(xs)

    if n == 1:
        qc.z(xs[0])
    else:
        qc.h(xs[-1])
        qc.mcx(xs[:-1], xs[-1])
        qc.h(xs[-1])

    qc.x(xs)
    qc.h(xs)

    return qc.to_gate()

## Now we do the Grover's algorithm

In [72]:
oracle = subset_sum_oracle_weighted_adder(S, T)
total_qubits = oracle.num_qubits
diffusion = diffusion_on_first_n_qubits(n, total_qubits)

qc = QuantumCircuit(total_qubits)

# Uniform superposition over all subsets.
qc.h(range(n))

# Grover iterations with theoretical number of iterations.
N = 2 ** n
M = len(solutions)
num_iterations = max(1, floor((pi / 4) * sqrt(N / M)))

print("Total qubits:", total_qubits)
print("Grover iterations:", num_iterations)

for _ in range(num_iterations):
    qc.append(oracle, range(total_qubits))
    qc.append(diffusion, range(total_qubits))

print(qc.draw(output="text", fold=-1))


Total qubits: 9
Grover iterations: 1
     ┌───┐┌──────────────────┐┌────────────┐
q_0: ┤ H ├┤0                 ├┤0           ├
     ├───┤│                  ││            │
q_1: ┤ H ├┤1                 ├┤1           ├
     ├───┤│                  ││            │
q_2: ┤ H ├┤2                 ├┤2           ├
     └───┘│                  ││            │
q_3: ─────┤3                 ├┤3           ├
          │                  ││            │
q_4: ─────┤4 SubsetSumOracle ├┤4 Diffusion ├
          │                  ││            │
q_5: ─────┤5                 ├┤5           ├
          │                  ││            │
q_6: ─────┤6                 ├┤6           ├
          │                  ││            │
q_7: ─────┤7                 ├┤7           ├
          │                  ││            │
q_8: ─────┤8                 ├┤8           ├
          └──────────────────┘└────────────┘


## Reading probabilities with `Statevector`
For the notebook benchmark, we simulate the final quantum state with:

```python
statevector = Statevector.from_instruction(qc)
```

`Statevector` gives the full amplitude vector for all qubits in the circuit. The selector qubits are the first `n` qubits, so we sum probabilities over all basis states that share the same selector bits. This gives a marginal probability distribution over subsets, without adding measurement gates to the algorithmic circuit.


In [73]:
def selector_probabilities(statevector: Statevector, n: int):
    """Marginalize full statevector probabilities onto selector qubits."""
    probabilities = {}
    for basis_index, probability in enumerate(statevector.probabilities()):
        bits = tuple((basis_index >> i) & 1 for i in range(n))
        probabilities[bits] = probabilities.get(bits, 0.0) + float(probability)
    return probabilities


statevector = Statevector.from_instruction(qc)
probabilities = selector_probabilities(statevector, n)
ranked = sorted(probabilities.items(), key=lambda item: item[1], reverse=True)

print("Top selector probabilities in x[0], x[1], x[2] order:")
for bits, probability in ranked[:8]:
    chosen = [int(S[i]) for i, bit in enumerate(bits) if bit]
    print(f"  bits={bits}  subset={chosen}  sum={sum(chosen)}  probability={probability:.6f}")

print("\nExpected marked bitstrings:")
print(solutions)


Top selector probabilities in x[0], x[1], x[2] order:
  bits=(1, 1, 0)  subset=[1, 2]  sum=3  probability=0.500000
  bits=(0, 0, 1)  subset=[3]  sum=3  probability=0.500000
  bits=(1, 0, 0)  subset=[1]  sum=1  probability=0.000000
  bits=(0, 1, 1)  subset=[2, 3]  sum=5  probability=0.000000
  bits=(0, 0, 0)  subset=[]  sum=0  probability=0.000000
  bits=(1, 1, 1)  subset=[1, 2, 3]  sum=6  probability=0.000000
  bits=(0, 1, 0)  subset=[2]  sum=2  probability=0.000000
  bits=(1, 0, 1)  subset=[1, 3]  sum=4  probability=0.000000

Expected marked bitstrings:
[(0, 0, 1), (1, 1, 0)]
